In [ ]:
from pathlib import Path
import sys

import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'shared').exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'shared').exists() and (candidate / 'artifacts').exists():
            PROJECT_ROOT = candidate
            break

SRC_DIR = PROJECT_ROOT / 'artifacts' / 'ps-004-headcount-forecasting' / 'src'
MODELS_SRC_DIR = SRC_DIR / 'models'
for path in (SRC_DIR, MODELS_SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from feature_engineering import RAW_INPUT_PATH, OUTPUT_PATH, aggregate_profession_totals, build_features
from linear_model import DATE_STAMP, MODELS_DIR, train_linear_models

In [ ]:
raw_df = pl.read_parquet(RAW_INPUT_PATH)
all_sector_df = raw_df.filter(pl.col('sector') == 'All')
print({'raw_shape': raw_df.shape, 'all_sector_shape': all_sector_df.shape})
profession_totals = aggregate_profession_totals(raw_df)
print({'profession_totals_shape': profession_totals.shape})
profession_totals

In [ ]:
features = build_features(profession_totals)
features.head(10)

In [ ]:
models = train_linear_models(features)
for profession, model in models.items():
    print(profession, {'coefficient': float(model.coef_[0]), 'intercept': float(model.intercept_)})

In [ ]:
saved_features = pl.read_parquet(OUTPUT_PATH)
print({'features_shape': saved_features.shape})
saved_features.head(10)

In [ ]:
model_files = sorted(path.name for path in MODELS_DIR.glob(f'*_linear_{DATE_STAMP}.pkl'))
model_files